# Legacy water geometry optimization with CMA-ES and Gaussian

This small example evaluates every candidate geometry with a Gaussian
`B3LYP/6-31G*` single-point calculation. The two O-H bonds are constrained
to have the same length, leaving one bond length and one H-O-H angle to optimize.
Lengths are in Angstrom, angles in degrees, and energies in Hartree.

Install the Python dependencies using `requirements.txt`. Gaussian must be
installed and configured separately. See the [README](../../README.md),
[Chinese README](../../README.zh-CN.md), and [experiment background](../../docs/experiment.md).

The primary project version uses PySCF. See [legacy setup instructions](README.md) for this Gaussian notebook.

## 1. Imports and calculation settings

In [ ]:
import math
from pathlib import Path
import re
import shutil
import subprocess
import warnings

import cma


In [ ]:
nproc = 4
mem = "4GB"
energy_keyword = "# B3LYP/6-31G*"
charge = 0
multiplicity = 1
external_keyword = ""
gaussian_command = "g09"  # Executable name or path; use g16 if appropriate.
gaussian_workdir = Path("gaussian_work")
calculation_timeout = 120  # Seconds per single-point calculation.
penalty = 999.999  # Hartree; failed candidates must not look like low energies.

init_coord = [0.9584, 104.45]  # Shared O-H length (Angstrom), H-O-H angle (degrees).
angle_scale = 100.0
sigma0 = 0.2
seed = 2445
maxiter = 80


## 2. Gaussian energy evaluation

Each candidate overwrites `gaussian_work/energy.gjf` and `energy.log`.
Calculations are serial. The subprocess exit status and the final normal
termination marker are checked before the last matching archive energy is used.
The default `HF=` label is used by Gaussian for HF and DFT total energies;
other methods may require a different archive label.

In [ ]:
def energy_calculate(atom_list, energy_keyword, charge=0, multiplicity=1, external_keyword=""):
    gaussian_workdir.mkdir(parents=True, exist_ok=True)
    input_path = gaussian_workdir / "energy.gjf"
    log_path = gaussian_workdir / "energy.log"
    input_text = (
        f"%nprocshared={nproc}\n%mem={mem}\n{energy_keyword}\n\n"
        f"energy_calculation\n\n{charge} {multiplicity}\n"
        + "\n".join(atom_list)
        + f"\n\n{external_keyword}\n\n"
    )
    input_path.write_text(input_text, encoding="utf-8")
    with input_path.open("r", encoding="utf-8") as input_file, log_path.open("w", encoding="utf-8") as log_file:
        subprocess.run(
            [gaussian_command],
            stdin=input_file,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            cwd=gaussian_workdir,
            timeout=calculation_timeout,
            check=True,
        )
    return log_path


def energy_extract(energy_label="HF=", filename=None):
    log_path = Path(filename) if filename is not None else gaussian_workdir / "energy.log"
    text = log_path.read_text(encoding="utf-8", errors="replace")
    normal = text.rfind("Normal termination of Gaussian")
    error = text.rfind("Error termination")
    if normal < 0 or error > normal:
        raise RuntimeError(f"Gaussian did not terminate normally; inspect {log_path}.")

    # Gaussian wraps archive records over lines; D exponents are also accepted.
    archive = re.sub(r"\s+", "", text)
    number = r"([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[EeDd][+-]?\d+)?)"
    matches = re.findall(r"\\" + re.escape(energy_label) + number + r"(?=\\)", archive)
    if not matches:
        raise ValueError(f"No {energy_label!r} archive energy found in {log_path}.")
    value = float(matches[-1].replace("D", "E").replace("d", "e"))
    if not math.isfinite(value):
        raise ValueError(f"Non-finite energy in {log_path}.")
    return value


In [ ]:
def energy(coord):
    bond = float(coord[0])
    angle = float(coord[1]) * angle_scale
    if not (math.isfinite(bond) and math.isfinite(angle) and bond > 0 and 0 < angle < 180):
        return penalty
    atom_list = ["O", f"H 1 {bond}", f"H 1 {bond} 2 {angle}"]
    try:
        log_path = energy_calculate(atom_list, energy_keyword, charge, multiplicity, external_keyword)
        return energy_extract(filename=log_path)
    except (subprocess.CalledProcessError, subprocess.TimeoutExpired, RuntimeError, ValueError) as exc:
        warnings.warn(f"Rejected Gaussian calculation: {exc}", RuntimeWarning, stacklevel=2)
        return penalty


## 3. Check the installation and initial geometry

Resolve the executable before changing into the calculation directory.
A missing executable raises an error immediately. A failed initial calculation
also stops the notebook so installation or input problems can be fixed first.

In [ ]:
executable = shutil.which(gaussian_command)
if executable is None:
    raise FileNotFoundError(f"Gaussian executable {gaussian_command!r} was not found. Configure Gaussian and restart the kernel.")
gaussian_command = str(Path(executable).resolve())
initial_energy = energy([init_coord[0], init_coord[1] / angle_scale])
if initial_energy >= penalty:
    raise RuntimeError("The initial Gaussian calculation failed; inspect gaussian_work/energy.log.")
print(f"Initial energy: {initial_energy:.12f} Hartree")


## 4. CMA-ES optimization

Search `[bond_length, angle / 100]` to give the variables comparable scales.
The physical search bounds are 0.5-2.0 Angstrom and 30-170 degrees.
These bounds are specific to this example. The default budget is 80 iterations.

In [ ]:
# Optimize the shared bond length and the scaled angle.
es = cma.CMAEvolutionStrategy(
    [init_coord[0], init_coord[1] / angle_scale],
    sigma0,
    {
        "seed": seed,
        "bounds": [[0.5, 30.0 / angle_scale], [2.0, 170.0 / angle_scale]],
        "maxiter": maxiter,
        "tolfun": 1e-8,
        "verb_log": 0,
        "verb_disp": 10,
    },
)
es.optimize(energy)


## 5. Result

In [ ]:
if es.result.xbest is None or not math.isfinite(es.result.fbest) or es.result.fbest >= penalty:
    raise RuntimeError("No successful energy evaluations; check the calculation settings and warnings.")

bond = float(es.result.xbest[0])
angle = float(es.result.xbest[1]) * angle_scale
print(f"Shared O-H bond: {bond:.8f} Angstrom")
print(f"H-O-H angle:    {angle:.8f} degrees")
print(f"Energy:         {es.result.fbest:.12f} Hartree")
print(f"Evaluations:    {es.result.evaluations}")
print(f"Stop reasons:   {dict(es.stop())}")


Check the stopping conditions: `maxiter` indicates that the iteration budget
was exhausted. For a separate Gaussian reference optimization, use
[`examples/water_reference.gjf`](examples/water_reference.gjf).
The original 2022 outputs are recorded in [the background document](../../docs/experiment.md).
This notebook contains no saved execution output; run it to obtain fresh results.